# Infra-Bench — Prithvi fine-tune at 0.3× Training Data

**Same protocol as the 1.0× version (`prithvi/ft_1.0x.ipynb`), but with a stratified
30% subsample of the TRAINING set only. Val and test sets are unchanged, so
metrics on this notebook are directly comparable to the 1.0× run.**

## What's different vs. the 1.0× notebook

- **Training set:** stratified 30% subsample (per-class proportions preserved).
- **Val set:** unchanged (same n).
- **Test set:** unchanged (same n).
- **All hyperparameters:** unchanged.
- **Output subdir:** `fm_eval_prithvi_finetune_0.3x_v1/`.
- **Aggregate JSON records:** `training_subsample_fraction`, `training_subsample_seed`, `training_subsample_note`.
- **Confusion-matrix figure title:** appended with " (0.3x train)".

## Subsampling determinism

`SUBSAMPLE_SEED = 42` (fixed) is used for the subsample construction, so all
3 training seeds see the same 30% of training data. This separates:
- seed variance (different head init + DataLoader shuffle order)
- subsample variance (would appear only if you changed SUBSAMPLE_SEED)

## What's carried over from the 1.0× notebook

- ASSET_TYPE_MAP legacy `water.treatment.plant → water.water_works` fix.
- Skip-if-per-seed-JSON-exists guard.
- Resume-from-checkpoint via per-epoch `checkpoint_final.pt`.
- Auto-skip smoke on resume (`AUTO_SKIP_SMOKE_IF_RESUMING`).
- Aggregate write guard (only writes 3-seed aggregate if SEEDS is full protocol).
- BEST_CKPT_BEFORE_TEST for test evaluation.

See the 1.0× notebook for full protocol details, hyperparameters, and
methodological notes.


In [ ]:
!pip install --upgrade --force-reinstall torch==2.6.0 torchvision==0.21.0 "pillow<12" -q
!pip install -q terratorch==0.99.8 transformers==4.41.0 huggingface-hub==0.36.2 pyarrow scikit-learn scipy
!pip install --upgrade --force-reinstall --no-deps scikit-learn scipy

In [ ]:
# pre-flight: check what's actually on disk for the critical libraries.
# pip queries the filesystem, not Python's in-memory imports, so this
# tells us the truth regardless of what's loaded.
import subprocess

critical = ['torch', 'torchvision', 'pillow', 'numpy', 'scipy',
            'scikit-learn', 'terratorch', 'transformers',
            'huggingface-hub']

for pkg in critical:
    result = subprocess.run(
        ['pip', 'show', pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        # parse the "Version: X.Y.Z" line
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                print(f'  {pkg:<20s} {line.split(":", 1)[1].strip()}')
                break
    else:
        print(f'  {pkg:<20s} NOT INSTALLED')

In [ ]:
# conditional numpy restore. terratorch==0.99.8 pip resolution may pin
# numpy down to 1.x on some Colab images, which then breaks sklearn 1.5+.
# only force the upgrade if we detect the downgrade. when we do upgrade,
# raise to force a clean kernel restart — pip cannot swap numpy in an
# already-loaded kernel.
import numpy as np
if np.__version__.startswith('1.'):
    print(f'numpy is on {np.__version__} (1.x). Restoring to 2.x and forcing a restart...')
    !pip install --upgrade --force-reinstall "numpy>=2.2"
    !pip install --upgrade --force-reinstall --no-deps scikit-learn scipy
    raise RuntimeError(
        'numpy was pinned to 1.x by the install above; restored to 2.x. '
        'Restart the Colab kernel (Runtime -> Restart session) and re-run '
        'from the top.'
    )
else:
    print(f'numpy {np.__version__} OK (no restore needed)')

# also re-verify sklearn/scipy import cleanly after the install step.
import sklearn, scipy
print(f'sklearn  {sklearn.__version__}')
print(f'scipy    {scipy.__version__}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Runtime -> Change runtime type -> GPU before training.')


In [ ]:
import torch
print(f"torch: {torch.__version__}")
print(f"torchvision available:", end=" ")
try:
    import torchvision
    print(f"{torchvision.__version__}")
except Exception as e:
    print(f"FAILED — {e}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
# verify nms operator (the one that was failing)
try:
    from torchvision.ops import nms
    print("torchvision.ops.nms: OK")
except Exception as e:
    print(f"torchvision.ops.nms: FAILED — {e}")

In [ ]:
# prithvi-EO-2.0-300M-TL is public, but logging in raises rate limits
# and avoids occasional 401s on first weight download. robust to a missing
# token — just logs a warning.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in to HuggingFace.')
else:
    print('No HF_TOKEN in Colab Secrets — proceeding anonymously. '
          'Add HF_TOKEN if you hit a 401 on the weight download.')


In [ ]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# only load_split_artifact is needed at training time. if the curation
# zip predates Phase 1, fall back to an inline definition.
try:
    from curation.utils.spatial_blocking import load_split_artifact
    print('Imported load_split_artifact from curation.utils.spatial_blocking')
except ImportError as e:
    print(f'Could not import (zip is pre-Phase-1): {e}. Using inline fallback.')
    import pandas as pd
    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))


In [ ]:
import os
from pathlib import Path

DATASETS_DRIVE      = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL      = '/content/datasets'
OUTPUT_DIR          = f'{DRIVE_ROOT}/results/fm_eval_prithvi_finetune_0.3x_v1'
# the split artifact ships inside the code zip, so it resolves from the
# extracted repo. Drive stays a fallback for setups that still stage it there.
_drive_split = f'{DRIVE_ROOT}/data/spatial_split/asset_id_to_split_v1.parquet'
try:
    from curation.paths import SPLIT_ARTIFACT as _repo_split
    SPLIT_ARTIFACT_PATH = str(_repo_split) if _repo_split.exists() else _drive_split
except Exception:
    SPLIT_ARTIFACT_PATH = _drive_split
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.water_works',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':                  'water.water_works',   # legacy manifest tag
    'water.water_works':                      'water.water_works',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}

# Prithvi band remap: our .npy is [B04, B03, B02, B08, B8A, B11, B12, VV, VH] indices 0..8.
# Prithvi wants: [B02, B03, B04, B8A, B11, B12]  = indices [2, 1, 0, 4, 5, 6]
PRITHVI_BAND_INDICES = [2, 1, 0, 4, 5, 6]

PERC_LO, PERC_HI = 2.0, 98.0

# preserve the LP notebook's TerraTorch escape hatch. default True (TerraTorch
# is the canonical Prithvi loader path per IBM/NASA docs).
TRY_TERRATORCH = True

# ---- fine-tune hyperparameters (Jakubik et al. 2023 / TerraTorch configs) ----
IMAGE_SIZE                 = 224
FT_EPOCHS                  = 25
FT_BATCH_NOMINAL           = 16          # target effective batch (may split via accum at level 2)
FT_BACKBONE_LR             = 6e-5        # backbone base LR (LLRD applies per depth)
FT_HEAD_LR                 = 1e-3        # head LR (separate param group)
FT_WD                      = 0.05
FT_LLRD_GAMMA              = 0.75        # LR × γ^(max_depth - depth)
FT_WARMUP_FRACTION         = 0.05        # 5% of total effective steps
WEIGHT_CAP                 = 10.0
SEEDS                      = [314, 271, 161]
# the aggregate write below is gated on matching this exactly
FULL_PROTOCOL_SEEDS                      = [314, 271, 161]


# ---- stratified subsampling of TRAINING set (val + test unchanged) ----
# 0.3× subsample of the training set, preserving per-class proportions.
# SUBSAMPLE_SEED is fixed across the 3 training seeds so seed-variance and
# subsample-variance are separated cleanly.
SUBSAMPLE_FRACTION = 0.3
SUBSAMPLE_SEED     = 42

# resume-from-checkpoint (mirrors CROMA FT): detect existing checkpoint_final.pt
# and pick up from the recorded epoch. set False to disable.
RESUME_FROM_CHECKPOINT = True

# auto-skip smoke check on resume: when any full-run seed dir already contains
# checkpoint_final.pt, treat it as a reconnect after disconnect and bypass the
# 2-epoch smoke. set False to force smoke regardless.
AUTO_SKIP_SMOKE_IF_RESUMING = True

RUN_NAME_PREFIX            = 'prithvi_finetune_0.3x_v1'
CONFUSION_CMAP             = 'Greens'
CONFUSION_TITLE_PREFIX     = 'Prithvi fine-tune v1 (0.3x train)'

# ---- memory fallback control ----
# bump between smoke runs based on peak GPU vs runtime memory.
#   level 0 (default): bs=16, no grad ckpt              — try first
#   level 1:           bs=16, grad ckpt ON              — first fallback
#   level 2:           bs=8,  grad accum×2, grad ckpt ON  — final fallback
# level 2 preserves effective batch size 16 via gradient accumulation.
MEMORY_FALLBACK_LEVEL = 0

# derive per-level settings
if MEMORY_FALLBACK_LEVEL == 0:
    FT_BATCH               = 16
    FT_GRAD_ACCUM_STEPS    = 1
    FT_GRAD_CHECKPOINTING  = False
elif MEMORY_FALLBACK_LEVEL == 1:
    FT_BATCH               = 16
    FT_GRAD_ACCUM_STEPS    = 1
    FT_GRAD_CHECKPOINTING  = True
elif MEMORY_FALLBACK_LEVEL == 2:
    FT_BATCH               = 8
    FT_GRAD_ACCUM_STEPS    = 2
    FT_GRAD_CHECKPOINTING  = True
else:
    raise ValueError(f'MEMORY_FALLBACK_LEVEL must be 0, 1, or 2; got {MEMORY_FALLBACK_LEVEL}')

FT_EFFECTIVE_BATCH = FT_BATCH * FT_GRAD_ACCUM_STEPS
assert FT_EFFECTIVE_BATCH == 16, (
    f'Effective batch drifted from 16 ({FT_EFFECTIVE_BATCH}); '
    'check MEMORY_FALLBACK_LEVEL settings.'
)

FINETUNE_PROTOCOL = {
    'backbone_lr':      FT_BACKBONE_LR,       # base — actual per-depth LR varies via LLRD
    'head_lr':          FT_HEAD_LR,
    'weight_decay':     FT_WD,
    'scheduler':        'cosine_with_linear_warmup',
    'warmup_fraction':  FT_WARMUP_FRACTION,
    'llrd':             FT_LLRD_GAMMA,
    'autocast':         True,
    'grad_checkpointing': FT_GRAD_CHECKPOINTING,
    'grad_accum_steps': FT_GRAD_ACCUM_STEPS,
    'epochs':           FT_EPOCHS,
    'batch_size':       FT_BATCH,
    'effective_batch_size': FT_EFFECTIVE_BATCH,
    'memory_fallback_level': MEMORY_FALLBACK_LEVEL,
    'class_weight_cap': WEIGHT_CAP,
    'seeds':            list(SEEDS),
    'unfreeze_scope':   'all_backbone_params_LLRD_gamma_0.75',
    'runtime_gpu':      'A100 (Colab)',
    'reference':        'Jakubik et al. 2023 (Prithvi) + TerraTorch configs',
}


def set_seed(seed):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f'Output dir:            {OUTPUT_DIR}')
print(f'Split artifact:        {SPLIT_ARTIFACT_PATH}')
print(f'Prithvi band indices:  {PRITHVI_BAND_INDICES}  (B02, B03, B04, B8A, B11, B12)')
print(f'Training seeds:        {SEEDS}')
print(f'Fine-tune protocol:    {FT_EPOCHS} epochs, effective batch {FT_EFFECTIVE_BATCH}')
print(f'  backbone_base_lr:    {FT_BACKBONE_LR}   (LLRD γ={FT_LLRD_GAMMA})')
print(f'  head_lr:             {FT_HEAD_LR}   (separate param group)')
print(f'  weight_decay:        {FT_WD}')
print(f'  scheduler:           cosine + {FT_WARMUP_FRACTION*100:.0f}% warmup')
print(f'  autocast:            fp16 (train only; eval stays fp32)')
print(f'MEMORY_FALLBACK_LEVEL: {MEMORY_FALLBACK_LEVEL}')
print(f'  batch_size:          {FT_BATCH}')
print(f'  grad_accum_steps:    {FT_GRAD_ACCUM_STEPS}')
print(f'  grad_checkpointing:  {FT_GRAD_CHECKPOINTING}')
print(f'Class weight cap:      {WEIGHT_CAP}')
print(f'TRY_TERRATORCH:        {TRY_TERRATORCH}')


In [ ]:
import zipfile, shutil, time, re
from pathlib import Path

drive_path = Path(DATASETS_DRIVE)
MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')


def discover_multisector_sources():
    sources, seen = [], set()
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m: continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS: continue
        key = (region, sector)
        if key in seen: continue
        seen.add(key)
        sources.append((region, sector, entry, entry.suffix == '.zip'))
    return sources


def materialize_source(region, sector, src_path, is_zip, force=False):
    folder_name = f'dataset_{region}_{sector}_v1_1k'
    target_dir  = Path(DATASETS_LOCAL) / folder_name
    if not force and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE]   {region:<22s} {sector:<10s} already present ({n} tiles)')
        return target_dir
    if not is_zip:
        if not (src_path / 'manifest.json').exists():
            return None
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(src_path, target_dir)
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [COPY]   {region:<22s} {sector:<10s} ({n} tiles)')
        return target_dir
    t0 = time.time()
    with zipfile.ZipFile(src_path) as zf:
        names = zf.namelist()
        wrapped_prefix = f'{folder_name}/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)
        if is_wrapped:
            zf.extractall(DATASETS_LOCAL)
        else:
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'  [EXTRACT]{region:<22s} {sector:<10s} {time.time()-t0:.0f}s ({n} tiles)')
    return target_dir


discovered = discover_multisector_sources()
ready = []
for region, sector, src_path, is_zip in discovered:
    local = materialize_source(region, sector, src_path, is_zip)
    if local and (local / 'manifest.json').exists() and any((local / 'images').glob('*.npy')):
        ready.append((region, sector, local))
print(f'\nReady: {len(ready)} (region, sector) pairs')


In [ ]:
import json
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from collections import Counter
import random


@dataclass
class _Record:
    path: 'Path'
    asset_id: str
    asset_type: str


def percentile_normalize(arr, lo=PERC_LO, hi=PERC_HI):
    img = arr.astype(np.float32)
    low = np.percentile(img, lo); high = np.percentile(img, hi)
    if high <= low:
        return np.clip(img / 255.0, 0.0, 1.0)
    return np.clip((img - low) / (high - low), 0.0, 1.0)


class PrithviDataset(Dataset):
    """Self-contained loader for a single `dataset_<region>_<sector>_v1_1k/`
    folder. Selects the 6 Prithvi bands per `PRITHVI_BAND_INDICES`, applies
    percentile_normalize, and unsqueezes a T=1 temporal dim so the default
    collate stacks to `(B, 6, 1, H, W)` (Prithvi's 3D patch embedding
    expects 5D input)."""
    def __init__(self, dataset_root,
                 band_indices=PRITHVI_BAND_INDICES,
                 allowed_asset_types=tuple(ASSET_TYPE_MAP.keys())):
        self.dataset_root = Path(dataset_root)
        self.band_indices = list(band_indices)
        self.allowed = set(allowed_asset_types)
        self.max_required_band = max(self.band_indices)

        manifest_path = self.dataset_root / 'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'missing manifest.json: {manifest_path}')
        with manifest_path.open() as f:
            manifest = json.load(f)
        records_in = manifest.get('records', [])
        images_dir = self.dataset_root / 'images'

        records, dropped = [], Counter()
        for r in records_in:
            at = r.get('asset_type')
            if not at or at not in self.allowed:
                dropped['filtered_type' if at else 'no_label'] += 1; continue
            img_file = r.get('image_file')
            if not img_file:
                dropped['no_image_file'] += 1; continue
            p = images_dir / img_file
            if not p.exists():
                dropped['missing_npy'] += 1; continue
            try:
                arr = np.load(p, mmap_mode='r')
                if arr.shape[0] < self.max_required_band + 1:
                    dropped['too_few_bands'] += 1; continue
            except Exception:
                dropped['load_error'] += 1; continue
            records.append(_Record(path=p, asset_id=str(r.get('asset_id', p.stem)),
                                   asset_type=at))
        if dropped:
            print(f'  PrithviDataset({self.dataset_root.name}): dropped '
                  f'{sum(dropped.values())} records ({dict(dropped)})')
        if not records:
            raise RuntimeError(f'no usable records in {self.dataset_root}')
        self.records = records

    def __len__(self): return len(self.records)

    def _load_image(self, path):
        arr = np.load(path)
        arr = arr[self.band_indices, :, :]              # (6, H, W) in Prithvi band order
        arr = percentile_normalize(arr)                  # -> [0, 1]
        return arr.astype(np.float32)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self._load_image(r.path)                   # (6, H, W)
        t = torch.from_numpy(img).unsqueeze(1)           # (6, 1, H, W)  — T=1 dim
        return {'image': t, 'asset_id': r.asset_id, 'asset_type': r.asset_type}


class MultiSectorLabelWrapper(Dataset):
    def __init__(self, base, region, sector, input_size=IMAGE_SIZE):
        self.base = base; self.region = region; self.sector = sector
        self.input_size = input_size
        self.valid_indices, self.labels, self.asset_ids = [], [], []
        for i, r in enumerate(base.records):
            mapped = ASSET_TYPE_MAP.get(r.asset_type)
            if mapped is None: continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])
            self.asset_ids.append(r.asset_id)

    def __len__(self): return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.base[self.valid_indices[idx]]
        img = sample['image']                            # (6, 1, H, W)
        C, T, H, W = img.shape
        img = img.reshape(C * T, 1, H, W)
        img = F.interpolate(img, size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False)
        img = img.reshape(C, T, self.input_size, self.input_size)
        return {'image': img, 'label': self.labels[idx],
                'asset_id': sample['asset_id'],
                'region': self.region, 'sector': self.sector}


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base = base; self.indices = indices
        self.region = base.region; self.sector = base.sector
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


source_datasets = {}
for region, sector, local in ready:
    base = PrithviDataset(local)
    ds = MultiSectorLabelWrapper(base, region=region, sector=sector)
    if len(ds): source_datasets[(region, sector)] = ds
print(f'Built {len(source_datasets)} cell datasets')


In [ ]:
# load the spatial split artifact and slice each cell into train/val/test.
asset_to_split = load_split_artifact(SPLIT_ARTIFACT_PATH)
print(f'Loaded split artifact: {len(asset_to_split):,} asset_id -> split entries')
print(f'  splits distribution: {Counter(asset_to_split.values())}')

splits = {}
n_unmapped = 0
for key, ds in source_datasets.items():
    region, sector = key
    tr_idx, va_idx, te_idx = [], [], []
    for i in range(len(ds)):
        asset_id = ds.asset_ids[i]
        sp = asset_to_split.get(asset_id)
        if sp == 'train':   tr_idx.append(i)
        elif sp == 'val':   va_idx.append(i)
        elif sp == 'test':  te_idx.append(i)
        else:                n_unmapped += 1
    splits[key] = {
        'train': SubsetView(ds, tr_idx),
        'val':   SubsetView(ds, va_idx),
        'test':  SubsetView(ds, te_idx),
    }

if n_unmapped > 0:
    print(f'NOTE: {n_unmapped} tiles have no split assignment (excluded from train/val/test).')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])

# ---- stratified 0.3× subsampling of TRAINING set (val + test unchanged) ----
# preserves per-class proportions. deterministic via SUBSAMPLE_SEED so all
# 3 training seeds see the same 30% subsample.
import random as _random
from collections import defaultdict as _defaultdict


class _StratifiedSubsample(Dataset):
    """Wraps a ConcatDataset with a subset of indices. Exposes `.labels`
    so the training-infra cell's `_labels_from(dataset)` (defined later)
    hits its `else` branch and reads labels cleanly."""
    def __init__(self, base, indices, labels):
        self.base = base
        self.indices = list(indices)
        self.labels = list(labels)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


def _iter_concat_labels(concat_ds):
    global_idx = 0
    for sub in concat_ds.datasets:
        if not isinstance(sub, SubsetView):
            raise ValueError(f'Unexpected sub-dataset type: {type(sub)}')
        for local_i in range(len(sub)):
            yield global_idx, sub.base.labels[sub.indices[local_i]]
            global_idx += 1


def _apply_stratified_subsample(dataset, fraction, seed):
    idx_labels = list(_iter_concat_labels(dataset))
    by_class = _defaultdict(list)
    for idx, label in idx_labels:
        by_class[label].append(idx)
    rng = _random.Random(seed)
    selected = []
    for label in sorted(by_class):
        class_idxs = by_class[label].copy()
        rng.shuffle(class_idxs)
        n_take = max(1, int(round(len(class_idxs) * fraction)))
        chosen = class_idxs[:n_take]
        selected.extend((i, label) for i in chosen)
    selected.sort()
    return _StratifiedSubsample(dataset,
                                 indices=[i for i, _ in selected],
                                 labels=[l for _, l in selected])


_orig_train_size = len(train_global)
train_global = _apply_stratified_subsample(
    train_global, SUBSAMPLE_FRACTION, SUBSAMPLE_SEED,
)
print(f'\nStratified {SUBSAMPLE_FRACTION:.1%} subsample of TRAINING set:')
print(f'  before: {_orig_train_size:>6d} tiles')
print(f'  after:  {len(train_global):>6d} tiles   '
      f'(seed={SUBSAMPLE_SEED}; val + test unchanged)')

from collections import Counter as _Counter
_class_counts = _Counter(train_global.labels)
print('  per-class counts (subsampled train):')
for _c in sorted(_class_counts):
    _name = CLASS_NAMES[_c] if _c < len(CLASS_NAMES) else f'class_{_c}'
    print(f'    [{_c:>2d}] {_name:<34s} {_class_counts[_c]:>4d}')

print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')


In [ ]:
# diagnostic: regenerate the v1 random stratified split inside the
# notebook for an exact-match comparison vs the new spatial split.
print('=' * 76)
print('Diagnostic: train/val/test transition table (old random -> new spatial)')
print('=' * 76)


def old_stratified_split(dataset_labels, train_frac=0.7, val_frac=0.15, seed=42):
    by_class = {}
    for i, label in enumerate(dataset_labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


old_split = {}
for key in sorted(source_datasets):
    ds = source_datasets[key]
    tr, va, te = old_stratified_split(ds.labels)
    for i in tr: old_split[ds.asset_ids[i]] = 'train'
    for i in va: old_split[ds.asset_ids[i]] = 'val'
    for i in te: old_split[ds.asset_ids[i]] = 'test'

common_ids = set(old_split) & set(asset_to_split)
print(f'Comparing on {len(common_ids):,} tiles in both old and new splits')
counts = Counter()
for aid in common_ids:
    counts[(old_split[aid], asset_to_split[aid])] += 1

print(f'\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ('train', 'val', 'test'):
    row = [counts.get((old_sp, new_sp), 0) for new_sp in ('train', 'val', 'test')]
    print(f'{old_sp:<8s}  {row[0]:>10d} {row[1]:>9d} {row[2]:>10d}')

same    = sum(counts.get((sp, sp), 0) for sp in ('train', 'val', 'test'))
changed = len(common_ids) - same
print(f'\nUnchanged: {same:,} ({100.0*same/len(common_ids):.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/len(common_ids):.1f}%)')
print('\n(Expected ~47% changed — same spatial split as CROMA v2 / AE v2 / SatlasS2 v2 / SatlasS1 v2.)')


In [ ]:
import torch.nn as nn


class PrithviBackbone(nn.Module):
    """Prithvi-EO-2.0-300M-TL frozen backbone, mean-pooled patch features.

    LOADER STRATEGY (v2 — TerraTorch primary)
    -----------------------------------------
    Per the IBM/NASA HuggingFace model card, TerraTorch is the canonical
    load path for Prithvi-EO-2.0. We try three TerraTorch entry points in
    order, then fall back to HF transformers if all three fail.

      1. `terratorch.registry.BACKBONE_REGISTRY.build('prithvi_eo_v2_300_tl', pretrained=True)`
         — the lowest-level primitive. Returns just the encoder.
      2. `terratorch.models.backbones.prithvi_select.prithvi_eo_v2_300_tl(pretrained=True)`
         — the timm-style factory function. Some terratorch versions expose
         only this path under `prithvi_select` (or `prithvi_vit`); the
         attempt below tries multiple submodule names.
      3. `terratorch.tasks.SemanticSegmentationTask(...).model.encoder`
         — build the full task wrapper (as the blumenstiel reference notebook
         does) then extract the encoder. Heavier but uses the high-level
         path the TerraTorch team specifically maintains.
      4. `transformers.AutoModel.from_pretrained(repo, trust_remote_code=True, num_labels=0)`
         — the v1 fallback. Currently known broken in this environment;
         included so the all-paths-failed diagnostic shows the full picture.

    If all four fail, we raise a single RuntimeError that lists each
    attempt's exception. The failure-pattern itself is the operative
    signal for whether Prithvi inclusion is feasible.

    FEATURE EXTRACTION (carried verbatim from v1)
    ---------------------------------------------
    Prithvi's encoder returns patch-token sequences `(B, N, D)`. We
    mean-pool over the token dim to get `(B, D)`. The forward is
    defensive against tuple/list/`ModelOutput`/3D/4D/5D return shapes.
    """
    NAME = 'prithvi_eo_v2_300m_tl'
    HF_REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
    EXPECTED_FEATURE_DIM = 1024

    def __init__(self, freeze=True):
        super().__init__()
        self._loader_used = None
        self._load_errors = []     # list of (attempt_name, exception_repr)
        self.backbone = self._load_backbone()
        self.feature_dim = self._infer_feature_dim()
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    # ------------------------------------------------------------------ #
    # loader attempts
    # ------------------------------------------------------------------ #

    def _try_backbone_registry(self):
        from terratorch.registry import BACKBONE_REGISTRY
        bb = BACKBONE_REGISTRY.build('prithvi_eo_v2_300_tl', pretrained=True)
        return bb

    def _try_timm_style_factory(self):
        # several terratorch versions stash the timm-style factory under
        # different submodules. try the common ones.
        candidate_modules = [
            'terratorch.models.backbones.prithvi_select',
            'terratorch.models.backbones.prithvi_vit',
            'terratorch.models.backbones',
        ]
        last_err = None
        for mod_path in candidate_modules:
            try:
                mod = __import__(mod_path, fromlist=['prithvi_eo_v2_300_tl'])
                factory = getattr(mod, 'prithvi_eo_v2_300_tl', None)
                if factory is None:
                    last_err = AttributeError(
                        f'{mod_path} has no prithvi_eo_v2_300_tl attribute'
                    )
                    continue
                return factory(pretrained=True)
            except Exception as e:
                last_err = e
                continue
        if last_err is not None:
            raise last_err
        raise RuntimeError('No candidate module exposed prithvi_eo_v2_300_tl')

    def _try_segmentation_task_extract(self):
        import terratorch
        # build the full task (matching the blumenstiel reference's pattern)
        # then extract the encoder. UNetDecoder + 2-class head is arbitrary;
        # we discard everything except the encoder.
        task = terratorch.tasks.SemanticSegmentationTask(
            model_factory='EncoderDecoderFactory',
            model_args={
                'backbone': 'prithvi_eo_v2_300_tl',
                'backbone_pretrained': True,
                'decoder': 'UNetDecoder',
                'decoder_channels': [256, 128, 64, 32],
                'num_classes': 2,
            },
            loss='ce',
            ignore_index=-1,
        )
        # probe likely attribute paths for the encoder.
        candidates = [
            ('task.model.encoder',  lambda: task.model.encoder),
            ('task.encoder',        lambda: task.encoder),
            ('task.model.backbone', lambda: task.model.backbone),
            ('task.model',          lambda: task.model),
        ]
        for label, getter in candidates:
            try:
                obj = getter()
                if obj is None:
                    continue
                # sanity-check: does it look like a Prithvi encoder?
                # just confirm it has parameters (a real nn.Module).
                if sum(p.numel() for p in obj.parameters()) > 1e5:
                    print(f'    extracted encoder via {label}')
                    return obj
            except AttributeError:
                continue
        # if nothing matched, list what's on task.model so the diagnostic
        # is informative.
        attrs = [a for a in dir(task.model) if not a.startswith('_')]
        raise RuntimeError(
            f'SemanticSegmentationTask built OK but no encoder attribute '
            f'found via {[c[0] for c in candidates]}. task.model attributes: '
            f'{attrs[:30]}{"..." if len(attrs) > 30 else ""}'
        )

    def _try_transformers_fallback(self):
        from transformers import AutoModel
        return AutoModel.from_pretrained(
            self.HF_REPO,
            trust_remote_code=True,
            num_labels=0,
        )

    def _load_backbone(self):
        attempts = [
            ('terratorch_registry',         self._try_backbone_registry),
            ('terratorch_timm_factory',     self._try_timm_style_factory),
            ('terratorch_task_extract',     self._try_segmentation_task_extract),
            ('transformers_automodel',      self._try_transformers_fallback),
        ]
        for name, fn in attempts:
            print(f'  attempt {name} ...', flush=True)
            try:
                bb = fn()
                print(f'    SUCCESS via {name}')
                self._loader_used = name
                return bb
            except Exception as e:
                msg = f'{e.__class__.__name__}: {e}'
                print(f'    FAILED: {msg}')
                self._load_errors.append((name, msg))

        # all four failed — surface the full picture.
        lines = ['', '=' * 76,
                 'Prithvi backbone load FAILED — all four loader paths exhausted',
                 '=' * 76]
        for name, msg in self._load_errors:
            lines.append(f'  [{name}]')
            for ln in msg.splitlines():
                lines.append(f'    {ln}')
        lines.extend([
            '',
            'Diagnostic next steps:',
            '  - If terratorch_registry / timm_factory / task_extract all hit the same',
            '    scipy/dask error: TerraTorch 0.99.8 is broken in this Colab image.',
            '    Either pin to a different terratorch version or defer Prithvi to v2 paper.',
            '  - If transformers_automodel hit num_labels: pin transformers==4.41.0 OR upgrade beyond.',
            '  - If HF Hub 401: huggingface-cli login or set HF_TOKEN.',
            '  - The error pattern above is the signal we need to decide whether',
            '    Prithvi inclusion is feasible for v1 paper scope.',
            '=' * 76,
        ])
        raise RuntimeError('\n'.join(lines))

    # ------------------------------------------------------------------ #
    # forward signature probing + feature extraction (unchanged from v1)
    # ------------------------------------------------------------------ #

    def _probe_forward(self, dummy):
        attempts = [
            lambda: self.backbone(dummy),
            lambda: self.backbone(dummy, temporal_coords=None, location_coords=None),
            lambda: self.backbone(pixel_values=dummy),
            lambda: self.backbone(pixel_values=dummy, temporal_coords=None, location_coords=None),
        ]
        last_err = None
        for attempt in attempts:
            try:
                return attempt()
            except (TypeError, RuntimeError) as e:
                last_err = e
        raise RuntimeError(f'No Prithvi forward signature worked. Last error: {last_err}')

    @staticmethod
    def _extract_features(out):
        if isinstance(out, (tuple, list)):
            out = out[-1]
        if hasattr(out, 'last_hidden_state'):
            out = out.last_hidden_state
        if out.dim() == 3:    return out.mean(dim=1)
        if out.dim() == 4:    return out.mean(dim=[2, 3])
        if out.dim() == 5:    return out.mean(dim=[2, 3, 4])
        raise RuntimeError(f'Unexpected Prithvi output shape: {tuple(out.shape)}')

    def _infer_feature_dim(self):
        self.backbone.eval()
        device = next(self.backbone.parameters()).device
        dummy = torch.zeros(1, 6, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
        with torch.no_grad():
            out = self._probe_forward(dummy)
            feat = self._extract_features(out)
        d = feat.shape[-1]
        if d != self.EXPECTED_FEATURE_DIM:
            print(f'  WARNING: feature_dim={d} (expected {self.EXPECTED_FEATURE_DIM} '
                  f'for 300M variant). Using observed dim.')
        else:
            print(f'  feature_dim = {d} (matches expected for Prithvi-EO-2.0 300M)')
        return d

    def forward(self, x):
        if x.dim() == 4:
            x = x.unsqueeze(2)
        out = self._probe_forward(x)
        return self._extract_features(out)


class InfraBenchClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(backbone.feature_dim, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


def BACKBONE_FACTORY(freeze=True):
    return PrithviBackbone(freeze=freeze)


print('PrithviBackbone + InfraBenchClassifier defined.')


In [ ]:
import re as _re
from torch.optim import AdamW
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set, max_weight=WEIGHT_CAP):
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
        'sector': [b['sector'] for b in batch],
    }


def _per_sector_v2(y_true, y_pred, per_class_f1, cm):
    """Per-sector F1 v2 — matches LP notebook exactly."""
    cm = np.array(cm)
    out = {}
    for sector, class_idxs in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_idxs]
        macro_f1 = float(np.mean(sector_f1s)) if sector_f1s else 0.0
        n_sector = int(sum(cm[i, :].sum() for i in class_idxs))
        correct  = int(sum(cm[i, i]      for i in class_idxs))
        acc = correct / n_sector if n_sector > 0 else float('nan')
        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_idxs)
            },
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    """Eval stays in fp32 to preserve F1 metric fidelity — autocast is
    training-only."""
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(len(CLASS_NAMES)))).tolist()
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion': cm,
    }
    if return_breakdowns:
        def grouped_region_f1():
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, all_regions):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region']                 = grouped_region_f1()
        result['per_sector']                 = _per_sector_v2(all_labels, all_preds, per_class_f1, cm)
        result['per_sector_corrected']       = result['per_sector']
        result['per_sector_correction_note'] = (
            'per_sector = macro-average of per-class F1s for classes in that '
            'sector, computed on the FULL test set (v2 definition, matches LP).'
        )
    return result


# ============================================================================
# LLRD (layer-wise LR decay) param-group builder
# ============================================================================
def build_llrd_param_groups(backbone, head, base_lr, head_lr, gamma, wd,
                             verbose=True):
    """Layer-wise LR decay for a ViT-like backbone + a separate head group.

    Depth convention: 0 = closest-to-input (smallest LR).
      - Non-block params (patch_embed, cls_token, pos_embed, norm before
        blocks) → depth 0.
      - blocks.N params → depth N + 1.
      - max_depth = max block index + 1.

    Per-param LR = base_lr × γ^(max_depth - depth).

    The head is placed in a distinct group with `head_lr` (does NOT get
    LLRD applied). Only trainable params are included.

    Returns
    -------
    (groups, diagnostics)
        groups: list[dict] suitable for AdamW(groups)
        diagnostics: list[(name, depth, lr)] for every backbone param
                     (for inspection in the smoke cell)
    """
    # discover block indices from backbone.named_parameters()
    block_pat = _re.compile(r'blocks?\.(\d+)\.')
    block_indices = set()
    for name, _ in backbone.named_parameters():
        m = block_pat.search(name)
        if m:
            block_indices.add(int(m.group(1)))
    if not block_indices:
        raise RuntimeError(
            'LLRD: no `blocks.N.` pattern found in backbone.named_parameters(). '
            'The TerraTorch loader may have returned a differently-named model. '
            'Inspect backbone.named_parameters() output and adjust the regex.'
        )
    max_block = max(block_indices)
    max_depth = max_block + 1   # +1 for the patch_embed/pre-block layers at depth 0

    # group params by depth
    per_depth = defaultdict(list)
    diagnostics = []
    for name, p in backbone.named_parameters():
        if not p.requires_grad:
            continue
        m = block_pat.search(name)
        depth = int(m.group(1)) + 1 if m else 0
        lr = base_lr * (gamma ** (max_depth - depth))
        per_depth[depth].append((name, p, lr))
        diagnostics.append((name, depth, lr))

    groups = []
    for depth in sorted(per_depth.keys()):
        entries = per_depth[depth]
        # all params at a given depth share the same LR by construction.
        lr = entries[0][2]
        groups.append({
            'params':       [p for (_, p, _) in entries],
            'lr':           lr,
            'weight_decay': wd,
            'name':         f'bb_depth{depth}',
        })
    # head: separate group, no LLRD
    head_params = [p for p in head.parameters() if p.requires_grad]
    groups.append({
        'params':       head_params,
        'lr':           head_lr,
        'weight_decay': wd,
        'name':         'head',
    })

    if verbose:
        print(f'  LLRD: {len(block_indices)} blocks, max_depth={max_depth}, '
              f'γ={gamma}, base_lr={base_lr}')
        print(f'  LLRD: {len(groups)} param groups (backbone: {len(groups)-1} × depth-buckets, '
              f'+ 1 head group at lr={head_lr})')
        # show LR range across depths
        depth_lrs = sorted(set((d, per_depth[d][0][2]) for d in per_depth), key=lambda x: x[0])
        print(f'  LLRD depth->lr: shallowest={depth_lrs[0][1]:.2e} (depth 0), '
              f'deepest={depth_lrs[-1][1]:.2e} (depth {max_depth})')

    return groups, diagnostics


def train_one_seed(seed, *, train_set, val_set, test_set,
                   num_epochs=None, scheduler_total_epochs=None,
                   allow_resume=None,
                   run_name_override=None):
    """Full fine-tune training loop for Prithvi (LLRD + autocast + optional
    grad accum + optional grad checkpointing).

    Structurally similar to S1/S2/CROMA fine-tune, with these changes:
      - LLRD param groups instead of a single AdamW(model.parameters(), lr)
      - Warmup fraction (FT_WARMUP_FRACTION) instead of a fixed step count
      - autocast + GradScaler in the training loop
      - Gradient accumulation if FT_GRAD_ACCUM_STEPS > 1
      - Optional gradient checkpointing (best-effort — depends on the
        TerraTorch loader path exposing gradient_checkpointing_enable())
      - Records llrd_depth_map in the result dict for the aggregate JSON
    """
    if num_epochs is None:
        num_epochs = FT_EPOCHS
    if scheduler_total_epochs is None:
        scheduler_total_epochs = num_epochs
    if allow_resume is None:
        allow_resume = RESUME_FROM_CHECKPOINT
    if allow_resume is None:
        allow_resume = RESUME_FROM_CHECKPOINT
    set_seed(seed)
    print(f'\n--- seed {seed}  fine-tune ---')
    print(f'    training loop: {num_epochs} epochs   scheduler length: {scheduler_total_epochs} epochs')
    print(f'    batch={FT_BATCH}  grad_accum={FT_GRAD_ACCUM_STEPS}  '
          f'effective_batch={FT_EFFECTIVE_BATCH}  grad_ckpt={FT_GRAD_CHECKPOINTING}')

    backbone = BACKBONE_FACTORY(freeze=False)
    model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Total params:     {total_params:>13,}')
    print(f'  Trainable params: {trainable_params:>13,}')

    # optional gradient checkpointing (best effort)
    grad_ckpt_active = False
    if FT_GRAD_CHECKPOINTING:
        target = backbone.backbone if hasattr(backbone, 'backbone') else backbone
        if hasattr(target, 'gradient_checkpointing_enable'):
            target.gradient_checkpointing_enable()
            grad_ckpt_active = True
            print('  gradient_checkpointing_enable() called on backbone')
        else:
            print('  WARNING: FT_GRAD_CHECKPOINTING=True but backbone lacks '
                  'gradient_checkpointing_enable(). Falling through with grad_ckpt OFF.')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_set, batch_size=FT_BATCH, shuffle=True,
                              num_workers=2, collate_fn=collate, pin_memory=True,
                              generator=g)
    val_loader   = DataLoader(val_set,   batch_size=FT_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=FT_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)

    # LLRD param groups
    param_groups, llrd_diag = build_llrd_param_groups(
        backbone=model.backbone, head=model.head,
        base_lr=FT_BACKBONE_LR, head_lr=FT_HEAD_LR,
        gamma=FT_LLRD_GAMMA, wd=FT_WD, verbose=True,
    )
    optimizer = AdamW(param_groups)

    # cosine + 5% warmup, per-*effective-step* stepping.
    steps_per_epoch_micro = len(train_loader)
    effective_steps_per_epoch = steps_per_epoch_micro // FT_GRAD_ACCUM_STEPS
    total_eff_steps = scheduler_total_epochs * effective_steps_per_epoch
    warmup_iters = max(1, int(FT_WARMUP_FRACTION * total_eff_steps))
    warmup_sched = LinearLR(optimizer, start_factor=1e-3, end_factor=1.0,
                             total_iters=warmup_iters)
    cosine_sched = CosineAnnealingLR(optimizer,
                                      T_max=max(total_eff_steps - warmup_iters, 1))
    scheduler    = SequentialLR(optimizer,
                                schedulers=[warmup_sched, cosine_sched],
                                milestones=[warmup_iters])

    scaler = GradScaler()

    run_name = run_name_override or f'{RUN_NAME_PREFIX}_seed{seed}_finetune'
    ckpt_dir = Path(OUTPUT_DIR) / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt     = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt    = ckpt_dir / 'checkpoint_final.pt'
    metrics_jsonl = ckpt_dir / 'metrics.jsonl'
    # resume-from-checkpoint (mirrors CROMA FT): if a prior session wrote
    # checkpoint_final.pt for this seed, load its state and continue from the
    # recorded epoch. otherwise fresh start + truncate metrics.jsonl.
    history, best_val_f1, best_epoch = [], -1.0, -1
    start_epoch = 0
    resumed_epochs = 0
    if allow_resume and final_ckpt.exists():
        ckpt = torch.load(final_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            try:
                scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            except NameError:
                pass  # notebook has no scheduler in scope
        if 'scaler_state_dict' in ckpt:
            try:
                scaler.load_state_dict(ckpt['scaler_state_dict'])
            except NameError:
                pass  # notebook has no GradScaler in scope
        history = ckpt.get('history', [])
        best_val_f1 = ckpt.get('best_val_f1', -1.0)
        best_epoch = ckpt.get('best_epoch', -1)
        start_epoch = int(ckpt.get('epoch', 0))
        resumed_epochs = start_epoch
        print(f'  [RESUME] Loaded {final_ckpt.name}: '
              f'{start_epoch}/{num_epochs} epochs done, '
              f'best_val_f1={best_val_f1:.4f} at epoch {best_epoch}')
    else:
        metrics_jsonl.write_text('', encoding='utf-8')
    t_seed_start = time.time()

    for epoch in range(start_epoch, num_epochs):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        optimizer.zero_grad()
        for micro_step, batch in enumerate(train_loader):
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            with autocast(dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)
            # scale loss by 1/accum so the accumulated gradient magnitude
            # matches an un-accumulated bs=FT_EFFECTIVE_BATCH step.
            scaler.scale(loss / FT_GRAD_ACCUM_STEPS).backward()
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
            if (micro_step + 1) % FT_GRAD_ACCUM_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()   # per-*effective-step* cosine step

        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)   # fp32 eval
        # LR snapshot: report the max across param groups (backbone deepest
        # + head are the two largest). full mapping saved to llrd_depth_map.
        current_lr = max(pg['lr'] for pg in optimizer.param_groups)
        entry = {
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'current_lr': current_lr,
            'time_s': time.time() - t0,
        }
        history.append(entry)
        with metrics_jsonl.open('a', encoding='utf-8') as f:
            f.write(json.dumps(entry) + '\n'); f.flush(); os.fsync(f.fileno())

        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch = epoch + 1
            marker = ' *'
            torch.save({'epoch': epoch + 1,
                        'model_state_dict': model.state_dict(),
                        'val_macro_f1': val['macro_f1'],
                        'history': history}, best_ckpt)
        # persist per-epoch checkpoint_final.pt for resume-safety on disconnect.
        _fckpt = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'best_val_f1': best_val_f1,
            'best_epoch': best_epoch,
        }
        try: _fckpt['scheduler_state_dict'] = scheduler.state_dict()
        except NameError: pass
        try: _fckpt['scaler_state_dict'] = scaler.state_dict()
        except NameError: pass
        torch.save(_fckpt, final_ckpt)
        # persist per-epoch checkpoint_final.pt for resume-safety on disconnect.
        _fckpt = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'best_val_f1': best_val_f1,
            'best_epoch': best_epoch,
        }
        try: _fckpt['scheduler_state_dict'] = scheduler.state_dict()
        except NameError: pass
        try: _fckpt['scaler_state_dict'] = scaler.state_dict()
        except NameError: pass
        torch.save(_fckpt, final_ckpt)
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  lr_max={current_lr:.2e}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    torch.save({'epoch': num_epochs,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history, 'best_val_f1': best_val_f1,
                'best_epoch': best_epoch}, final_ckpt)

    # ============== BEST_CKPT_BEFORE_TEST (preserved from LP) ==========
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'  [BEST_CKPT_BEFORE_TEST] restored epoch {ckpt["epoch"]} '
              f'(val_f1={ckpt["val_macro_f1"]:.4f}) before test')
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
    else:
        tested_with = 'final-epoch state'

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_breakdowns=True)

    wall_time_s = time.time() - t_seed_start
    peak_gpu_gb = (torch.cuda.max_memory_allocated(DEVICE) / 1e9
                   if torch.cuda.is_available() else 0.0)

    # compact LLRD depth map for the aggregate JSON: {depth: lr} only
    depth_lr_map = {}
    for name, depth, lr in llrd_diag:
        depth_lr_map.setdefault(depth, lr)

    return {
        'run_name':      run_name,
        'backbone':      backbone.NAME,
        'condition':     'full_finetune',
        'num_epochs':    num_epochs,
        'scheduler_total_epochs': scheduler_total_epochs,
        'seed':          seed,
        'best_val_f1':   best_val_f1,
        'best_epoch':    best_epoch,
        'tail_mean_f1':  float(np.mean(tail)),
        'tail_std_f1':   float(np.std(tail)),
        'history':       history,
        'resumed_epochs': int(resumed_epochs),
        'test':          test,
        'tested_with':   tested_with,
        'peak_gpu_gb':   float(peak_gpu_gb),
        'wall_time_s':   float(wall_time_s),
        'grad_ckpt_active': bool(grad_ckpt_active),
        'llrd_depth_lr_map': {int(d): float(lr) for d, lr in depth_lr_map.items()},
        'total_params':      int(total_params),
        'trainable_params':  int(trainable_params),
    }


print('Fine-tune training infrastructure ready.')
print('  - LLRD (γ=0.75) param groups + separate head group at head_lr=1e-3')
print('  - AdamW, wd=0.05')
print('  - Cosine + 5% linear warmup, per-effective-step stepping')
print('  - autocast(fp16) + GradScaler in training loop; eval stays fp32')
print(f'  - grad_accum={FT_GRAD_ACCUM_STEPS}, grad_ckpt={FT_GRAD_CHECKPOINTING}')
print('  - scheduler_total_epochs decoupled from num_epochs')
print('  - Tracks peak_gpu_gb + wall_time_s per seed')


In [ ]:
# ============================================================================
# SMOKE CHECK — 2 epochs on seed 314. see S1 notebook for the schedule-preview
# rationale (scheduler_total_epochs=FT_EPOCHS previews the real 25-epoch LR).
#
# prithvi-specific: prints the LLRD depth→LR mapping so you can eyeball
# it before committing to a full run. if the block-index regex didn't
# match the TerraTorch loader's naming, you'll see nonsense values here.
# ============================================================================
SMOKE_ONLY   = False
SMOKE_EPOCHS = 2
SMOKE_SEED   = SEEDS[0]


# ---- auto-skip smoke on resume ---------------------------------------------
# if any full-run seed dir already contains checkpoint_final.pt, this is a
# reconnect after a disconnect — running smoke would waste ~10 min on a
# distinct SMOKE_ ckpt dir before the multi-seed cell can pick up the
# interrupted run. bypass smoke and go straight to resume.
_full_run_ckpts_present = any(
    (Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{s}_finetune' / 'checkpoint_final.pt').exists()
    for s in SEEDS
)
if AUTO_SKIP_SMOKE_IF_RESUMING and _full_run_ckpts_present:
    print('=' * 76)
    print('AUTO_SKIP_SMOKE: existing full-run checkpoint_final.pt detected in one')
    print(f'or more of SEEDS={SEEDS} ckpt dirs. Interpreting this as a reconnect')
    print('after a disconnect — skipping the 2-epoch smoke check.')
    print()
    print('SMOKE_ONLY set to False; the multi-seed cell below will resume.')
    print('=' * 76)
    SMOKE_ONLY = False
    smoke_result = None
else:
    print(f'SMOKE_ONLY = {SMOKE_ONLY}')
    print(f'Smoke check: {SMOKE_EPOCHS} epochs on seed {SMOKE_SEED}')
    print(f'MEMORY_FALLBACK_LEVEL = {MEMORY_FALLBACK_LEVEL}  '
          f'(bs={FT_BATCH}, accum={FT_GRAD_ACCUM_STEPS}, grad_ckpt={FT_GRAD_CHECKPOINTING})')
    print(f'Scheduler built for FULL {FT_EPOCHS}-epoch protocol (previews real LR trajectory).')
    print('=' * 76)

    # ---- 1) Model instantiation + LLRD mapping diagnostic ----
    print('\n[1] Instantiate fine-tune model + inspect LLRD depth->LR mapping...')
    set_seed(SMOKE_SEED)
    sb = PrithviBackbone(freeze=False)
    sm = InfraBenchClassifier(sb, num_classes=len(CLASS_NAMES)).to(DEVICE)
    _total  = sum(p.numel() for p in sm.parameters())
    _train  = sum(p.numel() for p in sm.parameters() if p.requires_grad)
    print(f'  Total params:     {_total:>13,}')
    print(f'  Trainable params: {_train:>13,}   (expect ~300M for full Prithvi fine-tune)')
    assert _train > 290_000_000, (
        f'Trainable param count {_train:,} looks too small for full fine-tune; '
        f'expected ~300M. Backbone freeze may not have been disabled.'
    )

    # LLRD diagnostic — inspect BEFORE training
    smoke_groups, smoke_diag = build_llrd_param_groups(
        backbone=sm.backbone, head=sm.head,
        base_lr=FT_BACKBONE_LR, head_lr=FT_HEAD_LR,
        gamma=FT_LLRD_GAMMA, wd=FT_WD, verbose=True,
    )
    print('\n  LLRD depth->LR mapping (first block per depth-bucket, alphabetical by name):')
    seen_depths = set()
    for name, depth, lr in smoke_diag:
        if depth in seen_depths: continue
        seen_depths.add(depth)
        print(f'    depth={depth:>2d}  lr={lr:.3e}   e.g. `{name}`')
    # head group
    print(f'    HEAD           lr={FT_HEAD_LR:.3e}')

    # sanity: expected pattern is monotonically increasing LR with depth.
    depths_seen = sorted(set(d for _, d, _ in smoke_diag))
    lrs_by_depth = {d: [] for d in depths_seen}
    for _, d, lr in smoke_diag:
        lrs_by_depth[d].append(lr)
    # all params at a given depth should share the same LR.
    for d, lrs in lrs_by_depth.items():
        assert len(set(f'{lr:.3e}' for lr in lrs)) == 1, (
            f'LLRD group depth={d} has multiple LRs; grouping is buggy.'
        )
    # depths should be non-decreasing in LR (deeper block = closer to base_lr).
    lr_seq = [lrs_by_depth[d][0] for d in depths_seen]
    assert lr_seq == sorted(lr_seq), (
        f'LLRD LRs are not monotonic in depth: {lr_seq}. '
        f'Check the block-index regex.'
    )
    print('  [OK] LLRD monotonic across depths.')

    # ---- 2) Forward + backward pass sanity ----
    print('\n[2] Forward + backward + autocast + GradScaler sanity check...')
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)

    smoke_loader = DataLoader(train_global, batch_size=FT_BATCH, shuffle=True,
                              num_workers=0, collate_fn=collate)
    batch = next(iter(smoke_loader))
    img = batch['image'].to(DEVICE)
    lbl = batch['label'].to(DEVICE)
    weights = compute_class_weights(train_global).to(DEVICE)
    crit  = nn.CrossEntropyLoss(weight=weights)
    opt   = AdamW(smoke_groups)
    scaler_smoke = GradScaler()

    sm.train()
    opt.zero_grad()
    with autocast(dtype=torch.float16):
        logits = sm(img)
        loss = crit(logits, lbl)
    scaler_smoke.scale(loss).backward()
    scaler_smoke.step(opt)
    scaler_smoke.update()
    print(f'  One-step loss: {loss.item():.4f}  (finite: {torch.isfinite(loss).item()})')
    assert torch.isfinite(loss).item(), 'Loss is NaN/Inf on first step — abort.'

    if torch.cuda.is_available():
        peak_after_step = torch.cuda.max_memory_allocated(DEVICE) / 1e9
        print(f'  Peak GPU after 1 step: {peak_after_step:.2f} GB')
        # threshold depends on runtime GPU. A100 40 GB → warn at 35 GB; H100 80 GB → warn at 75.
        if peak_after_step > 35.0:
            print()
            print('!!' + '=' * 74)
            print(f'!! MEMORY WARNING: peak {peak_after_step:.2f} GB is above A100 40 GB comfort.')
            print(f'!! Current MEMORY_FALLBACK_LEVEL = {MEMORY_FALLBACK_LEVEL}.')
            if MEMORY_FALLBACK_LEVEL == 0:
                print('!! Recommended: bump MEMORY_FALLBACK_LEVEL to 1 (enable grad checkpointing)')
                print('!! and re-run this smoke cell.')
            elif MEMORY_FALLBACK_LEVEL == 1:
                print('!! Recommended: bump MEMORY_FALLBACK_LEVEL to 2 (bs=8 + accum×2, grad ckpt on)')
                print('!! and re-run this smoke cell.')
            elif MEMORY_FALLBACK_LEVEL == 2:
                print('!! Level 2 still hitting memory. Switch runtime to H100 for the full run.')
            print('!!' + '=' * 74)
        del sm, sb, opt, scaler_smoke

    # ---- 3) 2-epoch smoke run ----
    print(f'\n[3] Running {SMOKE_EPOCHS}-epoch smoke on seed {SMOKE_SEED} '
          f'(schedule built for {FT_EPOCHS} epochs)...')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)

    smoke_result = train_one_seed(
        SMOKE_SEED,
        train_set=train_global,
        val_set=val_global,
        test_set=test_global,
        num_epochs=SMOKE_EPOCHS,
        scheduler_total_epochs=FT_EPOCHS,
            run_name_override=f'{RUN_NAME_PREFIX}_SMOKE_seed{SMOKE_SEED}_finetune',
)

    hist = smoke_result['history']
    train_losses = [h['train_loss'] for h in hist]
    val_f1s      = [h['val_macro_f1'] for h in hist]
    per_epoch_s  = [h['time_s'] for h in hist]
    per_epoch_lr = [h['current_lr'] for h in hist]

    print('\n' + '=' * 76)
    print('SMOKE SUMMARY')
    print('=' * 76)
    print(f'  train_loss trajectory:   {[f"{l:.4f}" for l in train_losses]}')
    print(f'  val_macro_f1 trajectory: {[f"{f:.4f}" for f in val_f1s]}')
    print(f'  end-of-epoch lr_max:     {[f"{lr:.2e}" for lr in per_epoch_lr]}')
    print(f'  time per epoch:          {[f"{t:.0f}s" for t in per_epoch_s]}')
    print(f'  peak GPU during train:   {smoke_result["peak_gpu_gb"]:.2f} GB')
    print(f'  wall time (smoke seed):  {smoke_result["wall_time_s"]:.0f} s')
    print(f'  grad_ckpt_active:        {smoke_result["grad_ckpt_active"]}')

    # lr_max is the head_lr (1e-3) — it dominates and won't move much in 2 epochs
    # out of a 25-epoch schedule. if it dropped below FT_HEAD_LR × 0.5 the
    # scheduler is misbuilt.
    if per_epoch_lr[-1] < FT_HEAD_LR * 0.5:
        print()
        print('!!' + '=' * 74)
        print(f'!! LR SANITY: end-of-epoch-{SMOKE_EPOCHS} lr_max = {per_epoch_lr[-1]:.2e}')
        print(f'!! Expected close to FT_HEAD_LR={FT_HEAD_LR:.2e} with {FT_EPOCHS}-epoch schedule.')
        print('!!' + '=' * 74)

    avg_epoch_s = float(np.mean(per_epoch_s))
    extrapolated_per_seed_s = avg_epoch_s * FT_EPOCHS
    extrapolated_total_s    = extrapolated_per_seed_s * len(SEEDS)
    def _fmt_hms(s):
        s = int(round(s)); h, r = divmod(s, 3600); m, s = divmod(r, 60)
        return f'{h}h {m}m {s}s' if h else (f'{m}m {s}s' if m else f'{s}s')
    print(f'\n  Extrapolated per-seed:   {_fmt_hms(extrapolated_per_seed_s)}  '
          f'({FT_EPOCHS} epochs × {avg_epoch_s:.0f}s/epoch)')
    print(f'  Extrapolated total:      {_fmt_hms(extrapolated_total_s)}  '
          f'({len(SEEDS)} seeds × {_fmt_hms(extrapolated_per_seed_s)})')

    if len(train_losses) >= 2 and train_losses[-1] >= train_losses[0]:
        print()
        print('!!' + '=' * 74)
        print('!! WARNING: train_loss did NOT decrease over the smoke run.')
        print(f'!!   epoch 1 loss: {train_losses[0]:.4f}')
        print(f'!!   epoch {len(train_losses)} loss: {train_losses[-1]:.4f}')
        print('!!   For Prithvi FT: this is unusual with LLRD in place. Check')
        print('!!   (1) LLRD depth mapping above, (2) autocast NaN loss guard,')
        print('!!   (3) that WEIGHT_CAP=10 isn\'t overweighting minority classes')
        print('!!   with the larger effective gradients.')
        print('!!' + '=' * 74)
    else:
        if len(train_losses) >= 2:
            print(f'\n  Train loss decreased by {train_losses[0] - train_losses[-1]:.4f} '
                  f'over smoke.')
        print('  Ready to flip SMOKE_ONLY=False for the full 3-seed run.')

    smoke_out = Path(OUTPUT_DIR) / 'smoke_check_results.json'
    with smoke_out.open('w') as f:
        smoke_result_compact = dict(smoke_result)
        json.dump(smoke_result_compact, f, indent=2)
    print(f'\nSmoke results written: {smoke_out}')
    print(f'\nSMOKE_ONLY = {SMOKE_ONLY} — multi-seed cell below will '
          f'{"NOT train" if SMOKE_ONLY else "run all 3 seeds"}.')


In [ ]:
# ============================================================================
# multi-seed Prithvi fine-tune. same structure as S1/S2 fine-tune multi-seed,
# with the addition of llrd_depth_lr_map preserved in each seed's result
# and (from the last seed) in the aggregate JSON.
# ============================================================================
import json as _json
import numpy as np

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping the 3-seed fine-tune run.')
else:
    per_seed_results = {}
    for seed in SEEDS:
        out_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{seed}_results.json'
        if out_path.exists():
            # skip training: load existing per-seed JSON into per_seed_results
            # so the aggregate step below still works.
            # if a prior run was contaminated (e.g. by the water_works
            # ASSET_TYPE_MAP bug), delete the file on Drive to force a fresh train.
            with out_path.open() as f:
                per_seed_results[seed] = _json.load(f)['finetune']
            print(f'  [SKIP] seed {seed}: existing per-seed JSON at '
                  f'{out_path.name} (delete to force rerun)')
            continue
        result = train_one_seed(seed,
                                train_set=train_global,
                                val_set=val_global,
                                test_set=test_global)
        per_seed_results[seed] = result
        out_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{seed}_results.json'
        with open(out_path, 'w') as f:
            _json.dump({'finetune': result}, f, indent=2)
        print(f'\n  saved {out_path}')

    def _agg(values):
        arr = np.array(values, dtype=np.float64)
        return {'mean': float(arr.mean()), 'std': float(arr.std(ddof=0)),
                'per_seed': [float(v) for v in arr]}

    agg = {}
    agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in SEEDS])
    agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc'] for s in SEEDS])

    per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in SEEDS])
    agg['per_class_f1'] = [
        {'class': CLASS_NAMES[i], 'idx': i,
         'mean_f1':  float(per_class_arr[:, i].mean()),
         'std_f1':   float(per_class_arr[:, i].std(ddof=0)),
         'per_seed': [float(v) for v in per_class_arr[:, i]]}
        for i in range(len(CLASS_NAMES))
    ]

    agg['per_sector_f1'] = {}
    for sector in SECTOR_TO_CLASS_IDX:
        f1s = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1'] for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_sector'][sector]['n']
        agg['per_sector_f1'][sector] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.mean(f1s)),
            'std_macro_f1':  float(np.std(f1s, ddof=0)),
            'per_seed':      [float(v) for v in f1s],
        }

    region_keys = sorted({r for s in SEEDS for r in per_seed_results[s]['test']['per_region']})
    agg['per_region_f1'] = {}
    for region in region_keys:
        f1s = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
               for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
        agg['per_region_f1'][region] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.nanmean(f1s)),
            'std_macro_f1':  float(np.nanstd(f1s, ddof=0)),
            'per_seed':      [float(v) for v in f1s],
        }

    agg['seeds']                       = SEEDS
    agg['training_subsample_fraction'] = SUBSAMPLE_FRACTION
    agg['training_subsample_seed']     = SUBSAMPLE_SEED
    agg['training_subsample_note'] = (
        'Stratified subsample of the TRAINING set only; val + test '
        'sets unchanged. SUBSAMPLE_SEED=42 is fixed across the 3 '
        'training seeds so the reported std reflects seed variance only, '
        'not subsample variance.'
    )
    agg['split_artifact']              = SPLIT_ARTIFACT_PATH
    agg['per_sector_correction_note']  = per_seed_results[SEEDS[0]]['test']['per_sector_correction_note']

    agg['finetune_protocol'] = FINETUNE_PROTOCOL
    agg['peak_gpu_gb'] = {
        'mean':     float(np.mean([per_seed_results[s]['peak_gpu_gb'] for s in SEEDS])),
        'per_seed': {int(s): float(per_seed_results[s]['peak_gpu_gb']) for s in SEEDS},
    }
    agg['wall_time_s'] = {
        'mean':     float(np.mean([per_seed_results[s]['wall_time_s'] for s in SEEDS])),
        'total':    float(np.sum([per_seed_results[s]['wall_time_s'] for s in SEEDS])),
        'per_seed': {int(s): float(per_seed_results[s]['wall_time_s']) for s in SEEDS},
    }
    # from the last-trained seed
    agg['llrd_depth_lr_map'] = per_seed_results[SEEDS[-1]]['llrd_depth_lr_map']
    agg['grad_ckpt_active']  = per_seed_results[SEEDS[-1]]['grad_ckpt_active']

    # a partial rerun must not clobber a complete three-seed aggregate
    if set(SEEDS) == set(FULL_PROTOCOL_SEEDS):
        agg_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_aggregate.json'
        with open(agg_path, 'w') as f:
            _json.dump(agg, f, indent=2)
        print(f'\nAggregate saved: {agg_path}')
    else:
        print(f'\nNOTE: SEEDS = {SEEDS}, not the full protocol '
              f'{FULL_PROTOCOL_SEEDS}. Skipping the aggregate write '
              f'so any existing three-seed aggregate survives.')

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
    for s in SEEDS:
        cm_sum += np.array(per_seed_results[s]['test']['confusion'], dtype=np.int64)
    cm_norm = cm_sum.astype(np.float64)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_norm, row_sums, out=np.zeros_like(cm_norm), where=row_sums > 0)

    short_names = [n.split('.', 1)[1] if '.' in n else n for n in CLASS_NAMES]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, cmap=CONFUSION_CMAP, vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    mf1 = agg['test_macro_f1']
    ax.set_title(f'{CONFUSION_TITLE_PREFIX} — aggregate confusion (summed over '
                 f'{len(SEEDS)} seeds, row-normalized)\n'
                 f'macro F1 = {mf1["mean"]:.3f} +/- {mf1["std"]:.3f}, '
                 f'seeds = {SEEDS}')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', color=color, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    cm_path = Path(OUTPUT_DIR) / f'confusion_matrix_{RUN_NAME_PREFIX}_aggregate.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Confusion matrix saved: {cm_path}')

    def _fmt_hms(s):
        s = int(round(s)); h, r = divmod(s, 3600); m, s = divmod(r, 60)
        return f'{h}h {m}m {s}s' if h else (f'{m}m {s}s' if m else f'{s}s')
    print('\n' + '=' * 76)
    print(f'{CONFUSION_TITLE_PREFIX} — aggregate ({len(SEEDS)} seeds)')
    print('=' * 76)
    print(f'Test macro F1: {agg["test_macro_f1"]["mean"]:.4f} +/- {agg["test_macro_f1"]["std"]:.4f}')
    print(f'Test accuracy: {agg["test_accuracy"]["mean"]:.4f} +/- {agg["test_accuracy"]["std"]:.4f}')
    print(f'Peak GPU:      {agg["peak_gpu_gb"]["mean"]:.2f} GB (mean across seeds)')
    print(f'Wall clock:    per-seed {_fmt_hms(agg["wall_time_s"]["mean"])}, '
          f'total {_fmt_hms(agg["wall_time_s"]["total"])}')
    print(f'Grad ckpt:     {agg["grad_ckpt_active"]}')
    print('\nPer-class F1 (mean +/- std):')
    for entry in agg['per_class_f1']:
        print(f'  [{entry["idx"]:>2d}] {entry["class"]:<34s} '
              f'{entry["mean_f1"]:.4f} +/- {entry["std_f1"]:.4f}')
    print('\nPer-sector F1 (v2):')
    for sector, stats in agg['per_sector_f1'].items():
        print(f'  {sector:<10s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
    print('\nPer-region F1:')
    for region, stats in agg['per_region_f1'].items():
        print(f'  {region:<22s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
